# 05. Подготовка агрегатов для Yandex DataLens

##  Цель
Сформировать набор "плоских" CSV-файлов, которые позволят построить в DataLens **интерактивный дашборд** с кросс-фильтрацией.

## Логика подготовки
- Все агрегаты считаются **в PostgreSQL** — быстро, без выгрузки миллионов строк в pandas.
- pandas используется только как транспорт: `read_sql` → проверка → `to_csv`.
- Кодировка `utf-8-sig` — чтобы Excel и DataLens корректно читали кириллицу и специальные символы.

## Структура датасетов

| # | Файл | Назначение | Тип |
|---|------|------------|-----|
| 1 | `dl_kpi.csv` | KPI-карточки | Агрегат (1 строка) |
| 2 | `dl_by_year.csv` | Динамика по годам | Агрегат (год) |
| 3 | `dl_by_genre.csv` | Статистика по жанрам | Агрегат (жанр) |
| 4 | `dl_by_genre_combo.csv` | Комбо-жанры | Агрегат (пара жанров) |
| 5 | `dl_by_region.csv` | Статистика по регионам | Агрегат (регион) |
| 6 | `dl_region_genre.csv` | Связка регион + жанр | Агрегат (регион × жанр) |
| 7 | `dl_top_titles.csv` | Топ-100 фильмов | Детализация (фильм) |
| 8 | `dl_top_people.csv` | Топ людей | Агрегат (человек) |
| 9 | `dl_movie_people.csv` | **Фильм + человек + роль** | **Связка (для кликов)** |
| 10 | `dl_movie_regions.csv` | **Фильм + регион** | **Связка (для кликов)** |
| 11 | `dl_movie_genre.csv` | **Фильм + жанр** | **Связка (для кликов)** |
| 12 | `dl_content_types.csv` | Структура контента по типам | Агрегат (тип) |
| 13 | `dl_votes_buckets.csv` | Рейтинг и разброс по числу голосов | Агрегат (корзина голосов) |
| 14 | `dl_rating_votes.csv` | Рейтинг vs голоса, скрытые жемчужины | Детализация (фильм) |


In [34]:
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

DB_URL = (
    f"postgresql+psycopg://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL, connect_args={"options": "-csearch_path=imdb"})

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def export(df: pd.DataFrame, filename: str) -> None:
    path = PROCESSED_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{filename}: rows={len(df)}, cols={len(df.columns)}")


def check(df: pd.DataFrame, name: str) -> None:
    if df.empty:
        raise ValueError(f"{name} пустой")
    if df.isnull().sum().sum():
        print(f"{name}: есть NaN")

## KPI-карточки (`dl_kpi.csv`)

In [35]:
query_kpi = """
SELECT
    (SELECT COUNT(*) FROM title_basics) AS total_titles,
    (SELECT COUNT(*) FROM title_basics WHERE titletype = 'movie') AS total_movies,
    (SELECT COUNT(*) FROM title_basics WHERE titletype = 'tvSeries') AS total_series,
    (SELECT COUNT(*) FROM name_basics) AS total_people,
    (SELECT ROUND(AVG(averagerating)::numeric, 2) FROM title_ratings r JOIN title_basics b ON b.tconst = r.tconst WHERE b.titletype = 'movie') AS avg_movie_rating,
    (SELECT SUM(numvotes) FROM title_ratings r JOIN title_basics b ON b.tconst = r.tconst WHERE b.titletype = 'movie') AS total_movie_votes;
"""
df_kpi = pd.read_sql(query_kpi, engine)
check(df_kpi, "df_kpi")
export(df_kpi, "dl_kpi.csv")

dl_kpi.csv: rows=1, cols=6


## Динамика по годам (`dl_by_year.csv`)

Для линейного графика: сколько фильмов выходило каждый год + средний рейтинг + суммарные голоса.

**Фильтр:** `startyear BETWEEN 1900 AND 2025` — чтобы отсечь аномалии (единичные фильмы 1800-х и будущие релизы).
**Важно:** считаются только фильмы **с рейтингом** (`JOIN title_ratings`), поэтому цифры меньше, чем в `03_analysis.ipynb`, где считались все фильмы.

Добавлено поле `decade` (1990, 2000, …) для селектора по десятилетиям.

In [36]:
query_by_year = """
SELECT
    b.startyear AS year,
    (b.startyear / 10) * 10 AS decade,
    COUNT(DISTINCT b.tconst) AS movies_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating,
    SUM(r.numvotes) AS total_votes
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
WHERE b.titletype = 'movie' AND b.startyear BETWEEN 1900 AND 2025
GROUP BY b.startyear
ORDER BY b.startyear;
"""
df_by_year = pd.read_sql(query_by_year, engine)
check(df_by_year, "df_by_year")
export(df_by_year, "dl_by_year.csv")

KeyboardInterrupt: 

In [ ]:
df_by_year = pd.read_sql(query_by_year, engine)
check(df_by_year, "df_by_year")

print(f"Лет: {len(df_by_year)}")
print(f"Диапазон: {df_by_year['year'].min()}–{df_by_year['year'].max()}")
print(f"Пик: {df_by_year.loc[df_by_year['movies_count'].idxmax(), 'year']} "
      f"({df_by_year['movies_count'].max():,} фильмов)")
print(f"Средний рейтинг за всё время: {df_by_year['avg_rating'].mean():.2f}")
df_by_year.head()

Лет: 126
Диапазон: 1900–2025
Пик: 2023 (11,954 фильмов)
Средний рейтинг за всё время: 5.85


,year,decade,movies_count,avg_rating,median_rating,total_votes
0,1900,1900,6,3.72,3.6,179
1,1901,1900,8,3.58,3.6,189
2,1902,1900,3,2.53,2.6,50
3,1903,1900,4,4.23,3.9,921
4,1904,1900,3,4.47,5.3,624


## Статистика по жанрам (`dl_by_genre.csv`)

Для бар-чарта: сколько фильмов в каждом жанре + средний и медианный рейтинг.

**Фильтр:** `numvotes >= 6000` — чтобы отсечь фильмы с 1-2 голосами, которые искажают среднее.

**Порог:** `HAVING COUNT(DISTINCT b.tconst) >= 30`. Если в жанре остаётся один фильм, `STDDEV` возвращает `NULL`. Жанры с малым числом фильмов в любом случае статистически шумные.

Цифры (с порогом по голосам) не совпадают с `03_analysis.ipynb` — на дашборде подписывайте «фильмы с 6000+ голосов».

In [ ]:
query_by_genre = """
SELECT
    g.genre_name AS genre,
    COUNT(DISTINCT b.tconst) AS movies_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating,
    ROUND(STDDEV(r.averagerating)::numeric, 2) AS std_rating
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
JOIN title_genres tg ON tg.tconst = b.tconst
JOIN genres g ON g.genre_id = tg.genre_id
WHERE b.titletype = 'movie' AND r.numvotes >= 6000
GROUP BY g.genre_name
HAVING COUNT(DISTINCT b.tconst) >= 30
ORDER BY movies_count DESC;
"""
df_by_genre = pd.read_sql(query_by_genre, engine)
check(df_by_genre, "df_by_genre")
export(df_by_genre, "dl_by_genre.csv")  

dl_by_genre.csv: rows=22, cols=5


## Комбо-жанры (`dl_by_genre_combo.csv`)

Self-join `title_genres` с условием `g1.genre_id < g2.genre_id` — чтобы не задваивать пары и не строить (Drama, Drama).

**Фильтр:** `HAVING COUNT >= 100` — иначе в дашборде будут видны редкие комбо с шумным рейтингом.

In [ ]:
query_combo = """
SELECT
    g1.genre_name AS genre_1,
    g2.genre_name AS genre_2,
    g1.genre_name || ' + ' || g2.genre_name AS combo,
    COUNT(DISTINCT b.tconst) AS movies_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating,
    ROUND(STDDEV(r.averagerating)::numeric, 2) AS std_rating
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
JOIN title_genres tg1 ON tg1.tconst = b.tconst
JOIN title_genres tg2 ON tg2.tconst = b.tconst
JOIN genres g1 ON g1.genre_id = tg1.genre_id
JOIN genres g2 ON g2.genre_id = tg2.genre_id
WHERE b.titletype = 'movie' AND g1.genre_id < g2.genre_id
GROUP BY g1.genre_name, g2.genre_name
HAVING COUNT(DISTINCT b.tconst) >= 100
ORDER BY movies_count DESC;
"""
df_combo = pd.read_sql(query_combo, engine)
check(df_combo, "df_combo")
export(df_combo, "dl_by_genre_combo.csv")

dl_by_genre_combo.csv: rows=164, cols=7


## Статистика по регионам (`dl_by_region.csv`)

Из `title_akas` берём регион для каждого тайтла. Один тайтл может иметь несколько `region` — считаем все.

**Фильтр:** `HAVING COUNT >= 500` — чтобы убрать страны с 1-2 фильмами.

In [ ]:
query_by_region = """
SELECT
    ak.region,
    COUNT(DISTINCT b.tconst) AS titles_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
JOIN title_akas ak ON ak.titleid = b.tconst
WHERE b.titletype = 'movie' AND ak.region IS NOT NULL
GROUP BY ak.region
HAVING COUNT(DISTINCT b.tconst) >= 500
ORDER BY titles_count DESC;
"""
df_by_region = pd.read_sql(query_by_region, engine)
check(df_by_region, "df_by_region")
export(df_by_region, "dl_by_region.csv")

dl_by_region.csv: rows=85, cols=3


## Связка регион + жанр (`dl_region_genre.csv`)

Для тепловой карты: какие жанры популярны в каких регионах.

**Фильтр:** `HAVING COUNT >= 100` — чтобы не было шума.

In [ ]:
query_region_genre = """
SELECT
    ak.region,
    g.genre_name AS genre,
    COUNT(DISTINCT b.tconst) AS movies_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
JOIN title_akas ak ON ak.titleid = b.tconst
JOIN title_genres tg ON tg.tconst = b.tconst
JOIN genres g ON g.genre_id = tg.genre_id
WHERE b.titletype = 'movie' AND ak.region IS NOT NULL
GROUP BY ak.region, g.genre_name
HAVING COUNT(DISTINCT b.tconst) >= 100
ORDER BY ak.region, movies_count DESC;
"""
df_region_genre = pd.read_sql(query_region_genre, engine)
check(df_region_genre, "df_region_genre")
export(df_region_genre, "dl_region_genre.csv")

dl_region_genre.csv: rows=1539, cols=5


## Топ-100 фильмов (`dl_top_titles.csv`)

Топ по рейтингу с фильтром по голосам — чтобы в таблицу не попали «фильмы с 10.0 и 50 голосами».

**Порог:** `numvotes >= 100000`.

In [ ]:
query_top_titles = """
SELECT
    b.tconst, b.primarytitle AS title, b.startyear AS year,
    b.genres, b.runtimeminutes AS runtime,
    r.averagerating AS rating, r.numvotes AS votes
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
WHERE b.titletype = 'movie' AND r.numvotes >= 100000
ORDER BY r.averagerating DESC, r.numvotes DESC
LIMIT 100;
"""
df_top_titles = pd.read_sql(query_top_titles, engine)
check(df_top_titles, "df_top_titles")
export(df_top_titles, "dl_top_titles.csv")

dl_top_titles.csv: rows=100, cols=7


## Топ людей (`dl_top_people.csv`)

Два отдельных запроса → конкатенация в один датасет с колонкой `role`.

**Пороги:**
- Актёры: `numvotes >= 70000`, `movies_count >= 10`
- Режиссёры: `numvotes >= 70000`, `movies_count >= 5`

**Метрика:** `median_rating` — не среднее. Один провальный фильм не должен обрушивать медиану.

In [ ]:
query_top_people = """
WITH actors AS (
    SELECT n.primaryname AS person, 'actor' AS role, COUNT(DISTINCT b.tconst) AS movies_count,
           ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating
    FROM title_basics b
    JOIN title_ratings r ON r.tconst = b.tconst
    JOIN title_principals p ON p.tconst = b.tconst
    JOIN name_basics n ON n.nconst = p.nconst
    WHERE b.titletype = 'movie' AND r.numvotes >= 70000 AND p.category IN ('actor', 'actress')
    GROUP BY n.primaryname
    HAVING COUNT(DISTINCT b.tconst) >= 10
    ORDER BY median_rating DESC, movies_count DESC
    LIMIT 50
),
directors AS (
    SELECT n.primaryname AS person, 'director' AS role, COUNT(DISTINCT b.tconst) AS movies_count,
           ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating
    FROM title_basics b
    JOIN title_ratings r ON r.tconst = b.tconst
    JOIN title_principals p ON p.tconst = b.tconst
    JOIN name_basics n ON n.nconst = p.nconst
    WHERE b.titletype = 'movie' AND r.numvotes >= 70000 AND p.category = 'director'
    GROUP BY n.primaryname
    HAVING COUNT(DISTINCT b.tconst) >= 5
    ORDER BY median_rating DESC, movies_count DESC
    LIMIT 50
)
SELECT * FROM actors
UNION ALL
SELECT * FROM directors;
"""
df_top_people = pd.read_sql(query_top_people, engine)
check(df_top_people, "df_top_people")
export(df_top_people, "dl_top_people.csv")

dl_top_people.csv: rows=100, cols=5


## Связка фильм + человек (`dl_movie_people.csv`)

**Один из ключевых датасетов для интерактивности.**

Содержит:
- `tconst` и `title` — чтобы связать с таблицей «Топ фильмов»;
- `person` и `role` — актёры и режиссёры.

Связь с `dl_top_titles` делаем **по `tconst`**, а не по `title`: названия не уникальны (ремейки, одноимённые фильмы).

`role` нормализована (`actress` → `actor`), чтобы значения совпадали с `dl_top_people.csv` и один селектор «Роль» работал на оба датасета.

**Логика:** берём топ-100 фильмов и джойним с `title_principals`. Регионы вынесены в отдельный файл, чтобы не дублировать каждого человека для каждого региона.

In [ ]:
query_movie_people = """
WITH top_movies AS (
    SELECT b.tconst, b.primarytitle AS title, b.startyear AS year, r.averagerating AS rating
    FROM title_basics b
    JOIN title_ratings r ON r.tconst = b.tconst
    WHERE b.titletype = 'movie' AND r.numvotes >= 100000
    ORDER BY r.averagerating DESC, r.numvotes DESC
    LIMIT 100
)
SELECT DISTINCT
    tm.tconst, tm.title, tm.year, tm.rating,
    n.primaryname AS person,
    CASE WHEN p.category = 'actress' THEN 'actor' ELSE p.category END AS role
FROM top_movies tm
JOIN title_principals p ON p.tconst = tm.tconst
JOIN name_basics n ON n.nconst = p.nconst
WHERE p.category IN ('actor', 'actress', 'director')
ORDER BY tm.rating DESC, tm.title, role;
"""
df_movie_people = pd.read_sql(query_movie_people, engine)
check(df_movie_people, "df_movie_people")
export(df_movie_people, "dl_movie_people.csv")

dl_movie_people.csv: rows=1108, cols=6


## Связка фильм + регион (`dl_movie_regions.csv`)

Фильм × регион, связывается с `dl_top_titles` по `tconst`.

Напоминание: `title_akas.region` — страна, для которой есть альтернативное название, а не страна производства.

In [ ]:
query_movie_regions = """
WITH top_movies AS (
    SELECT b.tconst, b.primarytitle AS title, b.startyear AS year, r.averagerating AS rating
    FROM title_basics b
    JOIN title_ratings r ON r.tconst = b.tconst
    WHERE b.titletype = 'movie' AND r.numvotes >= 100000
    ORDER BY r.averagerating DESC, r.numvotes DESC
    LIMIT 100
)
SELECT DISTINCT
    tm.tconst, tm.title, tm.year, tm.rating, ak.region
FROM top_movies tm
JOIN title_akas ak ON ak.titleid = tm.tconst
WHERE ak.region IS NOT NULL
ORDER BY tm.rating DESC, tm.title, ak.region;
"""
df_movie_regions = pd.read_sql(query_movie_regions, engine)
check(df_movie_regions, "df_movie_regions")
export(df_movie_regions, "dl_movie_regions.csv")

dl_movie_regions.csv: rows=5678, cols=5


## Связка фильм + жанр (`dl_movie_genre.csv`)

Аналогично прошлому файлу, но для жанров. Позволит при клике на фильм фильтровать график по жанрам.

**Логика:** берём топ-100 фильмов и разворачиваем их жанры в отдельные строки.

In [ ]:
query_movie_genre = """
WITH top_movies AS (
    SELECT b.tconst, b.primarytitle AS title, b.startyear AS year, r.averagerating AS rating
    FROM title_basics b
    JOIN title_ratings r ON r.tconst = b.tconst
    WHERE b.titletype = 'movie' AND r.numvotes >= 100000
    ORDER BY r.averagerating DESC, r.numvotes DESC
    LIMIT 100
)
SELECT DISTINCT
    tm.tconst, tm.title, tm.year, tm.rating, g.genre_name AS genre
FROM top_movies tm
JOIN title_genres tg ON tg.tconst = tm.tconst
JOIN genres g ON g.genre_id = tg.genre_id
ORDER BY tm.rating DESC, tm.title;
"""
df_movie_genre = pd.read_sql(query_movie_genre, engine)
check(df_movie_genre, "df_movie_genre")
export(df_movie_genre, "dl_movie_genre.csv")

## Структура контента (`dl_content_types.csv`)

Для кольцевой диаграммы на странице «Общая картина»: 77% записей IMDb — эпизоды сериалов, фильмы — всего 6%.

В отличие от остальных датасетов, фильтра по `movie` здесь нет — цель показать структуру всей базы.

In [ ]:
query_content_types = """
SELECT
    titletype,
    COUNT(*) AS titles_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS share_pct
FROM title_basics
GROUP BY titletype
ORDER BY titles_count DESC;
"""
df_content_types = pd.read_sql(query_content_types, engine)
check(df_content_types, "df_content_types")
export(df_content_types, "dl_content_types.csv")

dl_content_types.csv: rows=11, cols=3


## Эффект голосов (`dl_votes_buckets.csv`)

Корзины и пороги те же, что в `04_advanced_analysis.ipynb`, чтобы цифры на дашборде совпадали с выводами.

Поле `bucket_order` нужно для сортировки: в DataLens текстовые подписи корзин иначе сортируются по алфавиту.

In [ ]:
query_votes_buckets = """
WITH num(bucket_order, votes_bucket, mini, maxi) AS (
VALUES
    (1, '< 1K',        100,     1000),
    (2, '1K - 5K',     1000,    5000),
    (3, '5K - 10K',    5000,    10000),
    (4, '10K - 50K',   10000,   50000),
    (5, '50K - 100K',  50000,   100000),
    (6, '100K - 500K', 100000,  500000),
    (7, '500K - 1M',   500000,  1000000),
    (8, '>= 1M',       1000000, 2147483647)
)
SELECT
    num.bucket_order,
    num.votes_bucket,
    COUNT(*) AS movies_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating,
    ROUND(STDDEV(r.averagerating)::numeric, 2) AS std_rating,
    MIN(r.averagerating) AS min_rating,
    MAX(r.averagerating) AS max_rating
FROM title_ratings r
JOIN title_basics b ON b.tconst = r.tconst
JOIN num ON r.numvotes >= num.mini AND r.numvotes < num.maxi
WHERE b.titletype = 'movie'
GROUP BY num.bucket_order, num.votes_bucket
ORDER BY num.bucket_order;
"""
df_votes_buckets = pd.read_sql(query_votes_buckets, engine)
check(df_votes_buckets, "df_votes_buckets")
export(df_votes_buckets, "dl_votes_buckets.csv")

dl_votes_buckets.csv: rows=8, cols=8


## Рейтинг vs голоса и «скрытые жемчужины» (`dl_rating_votes.csv`)

Один датасет для двух чартов: scatter «рейтинг vs голоса» (все фильмы с 1000+ голосов, цвет по `category`) и таблица жемчужин (фильтр `category = 'Жемчужины'`).

Категории как в ноутбуке 04:
- `Жемчужины`: рейтинг ≥ 8.0 и 5 000–50 000 голосов;
- `Звёзды (топ)`: рейтинг ≥ 8.0 и ≥ 100 000 голосов;
- `Остальные`: всё прочее.

`hidden_score = рейтинг × голоса / 1000` — индекс недооценённости.

In [ ]:
query_rating_votes = """
SELECT
    b.tconst,
    b.primarytitle AS title,
    b.startyear AS year,
    b.genres,
    b.runtimeminutes AS runtime,
    r.averagerating AS rating,
    r.numvotes AS votes,
    ROUND((r.averagerating * r.numvotes / 1000.0)::numeric, 1) AS hidden_score,
    CASE
        WHEN r.numvotes BETWEEN 5000 AND 50000 AND r.averagerating >= 8.0 THEN 'Жемчужины'
        WHEN r.numvotes >= 100000 AND r.averagerating >= 8.0 THEN 'Звёзды (топ)'
        ELSE 'Остальные'
    END AS category
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
WHERE b.titletype = 'movie' AND r.numvotes >= 1000;
"""
df_rating_votes = pd.read_sql(query_rating_votes, engine)
check(df_rating_votes, "df_rating_votes")
export(df_rating_votes, "dl_rating_votes.csv")

df_rating_votes: есть NaN
dl_rating_votes.csv: rows=49129, cols=9


## Финальная проверка

Убедимся, что все 14 файлов на месте и имеют корректный размер.

In [ ]:
expected = [
    "dl_kpi.csv", "dl_by_year.csv", "dl_by_genre.csv", "dl_by_genre_combo.csv",
    "dl_by_region.csv", "dl_region_genre.csv", "dl_top_titles.csv",
    "dl_top_people.csv", "dl_movie_people.csv", "dl_movie_regions.csv",
    "dl_movie_genre.csv", "dl_content_types.csv", "dl_votes_buckets.csv",
    "dl_rating_votes.csv",
]
for f in expected:
    path = PROCESSED_DIR / f
    if path.exists():
        print(f"OK {f} {path.stat().st_size / 1024:.1f} KB")
    else:
        print(f"MISSING {f}")

OK dl_kpi.csv 0.1 KB
OK dl_by_year.csv 4.0 KB
OK dl_by_genre.csv 0.6 KB
OK dl_by_genre_combo.csv 8.5 KB
OK dl_by_region.csv 1.2 KB
OK dl_region_genre.csv 38.2 KB
OK dl_top_titles.csv 6.8 KB
OK dl_top_people.csv 3.4 KB
OK dl_movie_people.csv 63.5 KB
OK dl_movie_regions.csv 229.3 KB
OK dl_movie_genre.csv 11.0 KB
OK dl_content_types.csv 0.3 KB
OK dl_votes_buckets.csv 0.4 KB
OK dl_rating_votes.csv 4319.3 KB
